# **Modelos ARCH para Previsão de Volatilidade de Ações da B3**

**Autor**: Jorge H. N. Viana

**Resumo**: Este notebook aplica modelos da família ARCH (*Autoregressive Conditional
Heteroskedasticity*) para modelar e prever a volatilidade dos retornos semanais de ações
listadas na B3 (Brasil, Bolsa, Balcão). Os dados de preços foram obtidos via Yahoo Finance
para um período de 2 anos, abrangendo aproximadamente 284 ativos.

Foram estimados e comparados quatro modelos:
1. **ARCH(1)** — modelo autorregressivo de heterocedasticidade condicional;
2. **GARCH(1,1)** — generalização que inclui a volatilidade passada;
3. **EGARCH(1,1,1)** — versão exponencial que captura assimetrias (efeito alavancagem);
4. **GJR-GARCH(1,1,1)** — modelo com termo assimétrico para choques negativos.

Adicionalmente, realiza-se validação cruzada com *grid search* para seleção ótima
de hiperparâmetros, e geram-se previsões de volatilidade para 5 semanas à frente.


# 1. CARREGAMENTO DOS PACOTES

Nesta seção são importadas as bibliotecas necessárias para a análise.
Destacam-se:
- `yfinance`: coleta de dados históricos de preços do Yahoo Finance;
- `arch`: biblioteca especializada para modelos ARCH/GARCH (funções `arch_model`,
  `GARCH`, `ConstantMean`, `Normal`);
- `joblib` e `multiprocessing`: paralelização dos ajustes para acelerar a validação cruzada;
- `utils.ajustar_modelo`: função auxiliar definida localmente para encapsular
  o ajuste de cada combinação de hiperparâmetros.

In [ ]:
# Pacotes
import numpy as np           
import pandas as pd           
import yfinance as yf  
from datetime import date, timedelta
from pathlib import Path
from multiprocessing import cpu_count
from tqdm import tqdm
import warnings
from arch.univariate import ConstantMean, Normal, GARCH, arch_model
from sklearn.metrics import mean_squared_error
import itertools
import random
from joblib import Parallel, delayed
from utils import ajustar_modelo


warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

## 2. CONFIGURAÇÕES

Definem-se aqui os parâmetros globais do notebook:
- **Período de análise**: 2 anos de dados (de meados de 2024 até maio de 2026);
- **Diretórios**: caminhos para dados de entrada (*Input*), dados intermediários
  (*Stage*), resultados (*Output*) e figuras (*Figures*);
- **Número de jobs**: utiliza todos os núcleos da CPU menos 2, para não sobrecarregar
  a máquina durante a validação cruzada paralelizada.

In [ ]:
# Define o período de tempo para coleta dos dados
# fim    = date.today()
fim    = date(2026, 5, 18)
inicio = fim - timedelta(days=365*2)

# Diretórios
dir_raiz   = "G:\\Meu Drive\\Ensino\\TAF\\taf-code"
input_dir  = Path(dir_raiz, 'Data\\Input\\')
stage_dir  = Path(dir_raiz, 'Data\\Stage\\')
output_dir = Path(dir_raiz, 'Data\\Output\\')
figure_dir = Path(dir_raiz, 'Figures\\')

# Jobs
jobs = cpu_count() - 2

# 3. EXTRAÇÃO DOS DADOS

Os preços das ações são obtidos do Yahoo Finance. O procedimento consiste em:
1. Carregar a lista de tickers de ações listadas na B3 a partir do arquivo CSV
   `acoes-listadas-b3.csv`;
2. Adicionar o sufixo `.SA` a cada ticker para compatibilidade com o Yahoo Finance;
3. Incluir o índice `^BVSP` (Ibovespa) como referência de mercado;
4. Baixar os preços ajustados (*Adj Close*) para o período definido;
5. Remover colunas (ativos) sem dados suficientes para a análise.

> **Nota**: Ativos que foram deslistados (*delisted*) durante o período geram falhas
> no download e são automaticamente excluídos da amostra.

### 3.1 Preços de Fechamento Ajustados

A obtenção dos dados brutos é feita via `yfinance.download()`, que retorna um DataFrame
com todos os preços ajustados de uma só vez. As colunas sem qualquer observação (ativos
que falharam no download) são removidas com `dropna(how='all', axis=1)`.

In [ ]:
# Todas as Empresas da B3
ativos = (pd.read_table(Path(input_dir, 
                             "acoes-listadas-b3.csv"), 
                        sep = ",")
          .sort_values(["Nome", "Negócios"], 
                       ascending= False)
          .drop_duplicates(["Nome"])
          .assign(Ticker = lambda x: x["Ticker"] + '.SA')
          ["Ticker"]
          .to_list()
          )
ativos.append('^BVSP')

# Coleta os dados do Yahoo Finance
dados = (yf
         .download(ativos, start=inicio, end=fim, 
                   threads = jobs,
                   auto_adjust=False)
         ['Adj Close']
         ).dropna(how = 'all', axis =1)

dados.head()

# 4. CÁLCULO DOS RETORNOS

Os retornos são calculados na frequência semanal a partir dos preços de fechamento
ajustados, utilizando a fórmula do log-retorno:

$$ r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) \times 100 $$

onde $P_t$ é o preço no fechamento da semana $t$.

O uso de retornos logarítmicos (em percentual) é padrão em econometria financeira porque:
- São aproximadamente simétricos e aditivos no tempo;
- Facilitam a modelagem da volatilidade condicional (heterocedasticidade);
- Tornam-se estacionários, condição necessária para modelos ARCH.

Além disso, são removidos ativos com menos de 100 observações válidas, garantindo
robustez amostral mínima para a estimação.

In [ ]:
log_retornos = ( (np.log(dados.resample('W').last()) 
                 - np.log(dados.resample('W').last().shift(1)) )
                 .dropna(thresh = 100, axis = 1)
                 .dropna(how ="all", axis = 1)          
                 .dropna()           
                 )*100

log_retornos.head()

# 5. MODELOS DE VOLATILIDADE CONDICIONAL

A família de modelos ARCH (Engle, 1982) e suas extensões modelam a volatilidade
(variância condicional) de séries temporais financeiras. A ideia central é que a
variância dos retornos no período $t$ não é constante, mas depende dos choques
passados (retornos quadráticos) e, em alguns modelos, da própria volatilidade
passada.

A seguir, cada modelo é apresentado da forma mais simples (estimação para uma única
ação) e depois generalizado para todos os ativos da amostra.

## 5.1 Modelo ARCH(1)

O modelo ARCH(1) (*Autoregressive Conditional Heteroskedasticity* de ordem 1)
especifica que a variância condicional no instante $t$ depende do quadrado do
choque (resíduo) no período imediatamente anterior:

$$ \sigma_t^2 = \omega + \alpha_1 \varepsilon_{t-1}^2 $$

onde:
- $\omega > 0$ é a constante (variância de longo prazo mínima);
- $\alpha_1 \geq 0$ mede o impacto de um choque recente na volatilidade atual;
- $\varepsilon_{t-1}$ é o resíduo (choque) do período anterior.

A equação da média é modelada como uma constante ($\mu$).

### 5.1.1 Especificação por componentes

O modelo ARCH(1) é construído manualmente, definindo cada componente separadamente:
média constante (`ConstantMean`), volatilidade GARCH com p=1 e q=0 (equivalente ao
ARCH(1)), e distribuição Normal para os erros. Esta abordagem permite maior controle
sobre a especificação e é útil para fins didáticos.

In [ ]:
am_1 = arch_model(log_retornos["AGRO3.SA"])

am_1.mean         = ConstantMean(log_retornos["AGRO3.SA"])
am_1.volatility   = GARCH(1,0,0)
am_1.distribution = Normal()

res_am_1 = am_1.fit(disp = 'off')   
print(res_am_1)

### 5.1.2 Especificação funcional

Utiliza-se a função `arch_model()` com argumentos nomeados para definir diretamente
o modelo. Esta é a forma mais concisa e recomendada para a maioria dos casos.

In [ ]:
# Inicialização funcional
am_1 = arch_model(log_retornos["AGRO3.SA"], 
                  mean = "Constant", 
                  vol = "ARCH", 
                  p = 1, 
                  dist = "normal")

# Estimação
res_am_1 = am_1.fit(update_freq = 0, 
                    disp = 'off')   
print(res_am_1)

### 5.1.3 Previsão de volatilidade

Após a estimação, geram-se previsões para um horizonte de 5 semanas via simulação
de Monte Carlo (100 simulações). As previsões incluem tanto os retornos médios
esperados quanto a variância condicional projetada para cada período futuro.

In [ ]:
forecasts = res_am_1.forecast(start=date.today(), 
                              method="simulation",
                              simulations = 100,
                              horizon=5)

forecast_retornos = forecasts.mean
forecast_variancia = forecasts.variance

print("Retoros Previstos\n", f'{forecast_retornos}\n','\n', 
      "Volatilidades previstas\n", forecast_variancia)

### 5.1.4 Aplicação a todos os ativos

O modelo ARCH(1) é estimado para cada uma das ~280 ações da amostra. Para cada
ativo, armazenam-se o retorno médio estimado ($\mu$) e as previsões de variância
para as 5 semanas seguintes. Os resultados são consolidados em DataFrames para
posterior comparação entre os ativos.

In [ ]:
resultados_arch = pd.DataFrame()
forecasts_arch  = pd.DataFrame()
for col in tqdm(log_retornos.columns):
    
    # Inicialização
    model = arch_model(log_retornos[col].dropna(), 
                      mean = "Constant", vol = "ARCH", 
                      p = 1, dist = "normal")

    # Estimação
    res_model = model.fit(update_freq = 0, disp = 'off')   
    
    # Previsões
    forecasts = res_model.forecast(start=date.today(), method="simulation", 
                                    simulations = 100,
                                      horizon=5)    
    
    # Armazennamento
    ## resultados
    resultados_temp = pd.DataFrame({
                                    'acao': [col],
                                    'r_medio': [res_model.params["mu"]],                       
                                    })
    ## volatilidade prevista
    variance_temp = (forecasts.variance
                     .assign(acao = col)
                     .reset_index(drop = True)
                     .melt(id_vars = "acao", value_name="v_prevista")
                     .drop(["variable"], axis = 1)
                     .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                     )
    ## retornos previstos
    returns_temp = (forecasts.mean
                    .assign(acao = col)
                    .reset_index(drop = True)
                    .melt(id_vars = "acao", value_name="r_previstos")
                    .drop(["variable"], axis = 1)
                    .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                    )
    ## Merge
    forecasts_temp = variance_temp.merge(returns_temp, how = "left")
    ## Concat
    forecasts_arch  = pd.concat([forecasts_temp, forecasts_arch ])
    resultados_arch = pd.concat([resultados_temp, resultados_arch])  


print("Resultados do Modelo ARCH(1) \n",  
      resultados_arch.sort_values(["r_medio"], ascending=False).head(10),
      "\n",
      "Previsões do Modelo ARCH(1)\n",
      forecasts_arch.sort_values(["r_previstos", "acao", "datas"], ascending=False).head(10)
      )

## 5.2 Modelo GARCH(1,1)

O GARCH(1,1) (Bollerslev, 1986) generaliza o ARCH ao incluir a volatilidade passada
como preditor da volatilidade atual:

$$ \sigma_t^2 = \omega + \alpha_1 \varepsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2 $$

A adição do termo $\beta_1$ (persistência) permite capturar a propriedade de
*volatility clustering* — períodos de alta volatilidade tendem a ser seguidos
por mais volatilidade alta — de forma mais parcimoniosa que um ARCH de ordem elevada.

A condição de estacionariedade é $\alpha_1 + \beta_1 < 1$.

In [ ]:
resultados_garch = pd.DataFrame()
forecasts_garch  = pd.DataFrame()
for col in tqdm(log_retornos.columns):
    
    # Inicialização
    model = arch_model(log_retornos[col].dropna(), 
                      mean = "Constant", 
                      vol = "GARCH", 
                      p = 1, q = 1, 
                      dist = "normal")

    # Estimação
    res_model = model.fit(update_freq = 0, disp = 'off')   
    
    # Previsões
    forecasts = res_model.forecast(start=date.today(), method="simulation", 
    simulations = 100,
                                      horizon=5)    
    
    # Armazennamento
    ## resultados
    resultados_temp = pd.DataFrame({
                                    'acao': [col],
                                    'r_medio': [res_model.params["mu"]],                       
                                    })
    ## volatilidade prevista
    variance_temp = (forecasts.variance
                     .assign(acao = col)
                     .reset_index(drop = True)
                     .melt(id_vars = "acao", value_name="v_prevista")
                     .drop(["variable"], axis = 1)
                     .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                     )
    ## retornos previstos
    returns_temp = (forecasts.mean
                    .assign(acao = col)
                    .reset_index(drop = True)
                    .melt(id_vars = "acao", value_name="r_previstos")
                    .drop(["variable"], axis = 1)
                    .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                    )
    ## Merge
    forecasts_temp = variance_temp.merge(returns_temp, how = "left")
    ## Concat
    forecasts_garch  = pd.concat([forecasts_temp, forecasts_garch ])
    resultados_garch = pd.concat([resultados_temp, resultados_garch])  


print("Resultados do Modelo GARCH(1) \n",  
      resultados_garch.sort_values(["r_medio"], ascending=False).head(10),
      "\n",
      "Previsões do Modelo GARCH(1)\n",
      forecasts_garch.sort_values(["r_previstos", "acao", "datas"], ascending=False).head(10)
      )

## 5.3 Modelo EGARCH(1,1,1)

O EGARCH — *Exponential GARCH* (Nelson, 1991) — modela o logaritmo da variância
condicional, o que garante automaticamente que $\sigma_t^2 > 0$ sem necessidade
de restrições de não negatividade nos parâmetros:

$$ \ln(\sigma_t^2) = \omega + \alpha_1 \left| \frac{\varepsilon_{t-1}}{\sigma_{t-1}} \right| + \gamma_1 \frac{\varepsilon_{t-1}}{\sigma_{t-1}} + \beta_1 \ln(\sigma_{t-1}^2) $$

O parâmetro $\gamma_1$ captura o **efeito alavancagem**: choques negativos
($\varepsilon_{t-1} < 0$) tendem a aumentar mais a volatilidade do que choques
positivos de mesma magnitude. Este é um fato estilizado importante em séries de
retornos de ações.

> **Nota**: A estimação do EGARCH é mais custosa computacionalmente e requer
> maior número de iterações (`maxiter=5000`).

In [ ]:
resultados_egarch = pd.DataFrame()
forecasts_egarch  = pd.DataFrame()
for col in tqdm(log_retornos.columns):
    
    # Inicialização
    model = arch_model(log_retornos[col].dropna(), 
                      mean = "Constant", 
                      vol = "EGARCH", 
                      p = 1, o = 1, q = 1,  
                      dist = "normal")

    # Estimação
    res_model = model.fit(update_freq = 0, options={'maxiter': 5000},
                          disp = 'off')   
    
    # Previsões
    forecasts = res_model.forecast(start=date.today(), method="simulation", simulations = 100,
                                      horizon=5)    
    
    # Armazennamento
    ## resultados
    resultados_temp = pd.DataFrame({
                                    'acao': [col],
                                    'r_medio': [res_model.params["mu"]],                       
                                    })
    ## volatilidade prevista
    variance_temp = (forecasts.variance
                     .assign(acao = col)
                     .reset_index(drop = True)
                     .melt(id_vars = "acao", value_name="v_prevista")
                     .drop(["variable"], axis = 1)
                     .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                     )
    ## retornos previstos
    returns_temp = (forecasts.mean
                    .assign(acao = col)
                    .reset_index(drop = True)
                    .melt(id_vars = "acao", value_name="r_previstos")
                    .drop(["variable"], axis = 1)
                    .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                    )
    ## Merge
    forecasts_temp = variance_temp.merge(returns_temp, how = "left")
    ## Concat
    forecasts_egarch  = pd.concat([forecasts_temp, forecasts_egarch ])
    resultados_egarch = pd.concat([resultados_temp, resultados_egarch ])  


print("Resultados do Modelo EGARCH(1) \n",  
      resultados_egarch .sort_values(["r_medio"], ascending=False).head(10),
      "\n",
      "Previsões do Modelo EGARCH(1)\n",
      forecasts_egarch .sort_values(["r_previstos", "acao", "datas"], ascending=False).head(10)
      )

## 5.4 Modelo GJR-GARCH(1,1,1)

O GJR-GARCH (Glosten, Jagannathan & Runkle, 1993) é outra extensão que captura
assimetria na volatilidade, mas o faz por meio de uma variável *dummy* para
choques negativos:

$$ \sigma_t^2 = \omega + \alpha_1 \varepsilon_{t-1}^2 + \gamma_1 \varepsilon_{t-1}^2 \cdot I(\varepsilon_{t-1} < 0) + \beta_1 \sigma_{t-1}^2 $$

O termo de alavancagem ($o=1$ no pacote `arch`) adiciona $\gamma_1 \varepsilon_{t-1}^2$
apenas quando $\varepsilon_{t-1} < 0$. Se $\gamma_1 > 0$, choques negativos amplificam
mais a volatilidade do que choques positivos, confirmando o efeito alavancagem.

Comparado ao EGARCH, o GJR-GARCH é computacionalmente mais simples e em muitos casos
empíricos oferece ajuste similar.

In [ ]:
resultados_gjrgarch = pd.DataFrame()
forecasts_gjrgarch  = pd.DataFrame()
for col in tqdm(log_retornos.columns):
    
    # Inicialização
    model = arch_model(log_retornos[col].dropna(), 
                      mean = "Constant", 
                      vol = "GARCH", 
                      p = 1, o = 1, q = 1, 
                      dist = "normal")

    # Estimação
    res_model = model.fit(update_freq = 0, disp = 'off')   
    
    # Previsões
    forecasts = res_model.forecast(start=date.today(), method="simulation", simulations = 100,
                                      horizon=5)    
    
    # Armazennamento
    ## resultados
    resultados_temp = pd.DataFrame({
                                    'acao': [col],
                                    'r_medio': [res_model.params["mu"]],                       
                                    })
    ## volatilidade prevista
    variance_temp = (forecasts.variance
                     .assign(acao = col)
                     .reset_index(drop = True)
                     .melt(id_vars = "acao", value_name="v_prevista")
                     .drop(["variable"], axis = 1)
                     .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                     )
    ## retornos previstos
    returns_temp = (forecasts.mean
                    .assign(acao = col)
                    .reset_index(drop = True)
                    .melt(id_vars = "acao", value_name="r_previstos")
                    .drop(["variable"], axis = 1)
                    .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                    )
    ## Merge
    forecasts_temp = variance_temp.merge(returns_temp, how = "left")
    ## Concat
    forecasts_gjrgarch  = pd.concat([forecasts_temp, forecasts_gjrgarch ])
    resultados_gjrgarch = pd.concat([resultados_temp, resultados_gjrgarch])  


print("Resultados do Modelo GARCH(1) \n",  
      resultados_gjrgarch.sort_values(["r_medio"], ascending=False).head(10),
      "\n",
      "Previsões do Modelo GARCH(1)\n",
      forecasts_gjrgarch.sort_values(["r_previstos", "acao", "datas"], ascending=False).head(10)
      )

# 6. VALIDAÇÃO CRUZADA

Para selecionar a melhor especificação de modelo para cada ação, realiza-se uma
busca em grade (*grid search*) sobre os hiperparâmetros, avaliando o RMSE
(*Root Mean Squared Error*) das previsões em um conjunto de teste.

O RMSE mede o erro quadrático médio entre os retornos previstos e os observados
no período de teste; quanto menor, melhor a capacidade preditiva do modelo.

## 6.1 Configuração do Grid Search

Define-se a grade de hiperparâmetros a serem testados:
- `p` (ARCH): 1 a 4 termos de choques passados;
- `o` (alavancagem): 1 a 2 termos assimétricos;
- `q` (GARCH): 1 a 4 termos de volatilidade passada;
- `lags` (AR na média): 0 a 4 defasagens autorregressivas na equação da média.

O produto cartesiano gera **todas as combinações possíveis** entre esses valores.
O conjunto de teste corresponde ao último terço da amostra (`n_teste`).

In [ ]:
# Definir os parâmetros a serem testados
hyper_grid = {
              'p': range(1,5),
              'o': range(1,3),
              'q': range(1,5),
              'lags': range(0,5)
             }
hyper_grid = list(itertools
                  .product(*hyper_grid.values()))

# Janela de teste
n_teste = len(log_retornos) // 3

## 6.2 Validação para uma única ação (AGRO3)

Aplica-se o método de validação (*grid search*) ao ativo AGRO3.SA como prova de
conceito. O código está comentado pois depende da execução prévia da célula de
configurações. O melhor modelo é selecionado com base no menor RMSE e utilizado
para gerar previsões otimizadas.

In [ ]:
# Treino e teste
treino = log_retornos["AGRO3.SA"][:-n_teste]
teste  = log_retornos["AGRO3.SA"][-n_teste:]

# Grid Search
## Inicialização
best_rmse = float('inf')
best_params = {}
metrics = pd.DataFrame()

for p, o, q, lags in tqdm(hyper_grid):

  # Ajustar o modelo ARCH
  am = arch_model(treino, vol='GARCH',
                  p=p, o = o, q=q, 
                  mean = "AR", lags = lags)
  res = am.fit(update_freq=5, disp='off')

  # Gerar previsões para teste
  previsoes = res.forecast(horizon=n_teste)

  # Calcular o RMSE nos dados de teste
  rmse = round(np.sqrt(mean_squared_error(teste, 
                          previsoes.mean.values[-1])), 6)
  
  # Armazenar métricas
  metrics_temp = pd.DataFrame({"p": [p], "o": [o], "q": [q], 
                               "lags": [lags], 
                               "rmse": [rmse]})
  metrics      = pd.concat([metrics, metrics_temp])

  if rmse < best_rmse:
    best_rmse = rmse
    best_params = {'p': p, 'q': q, 'lags': lags}

metrics = metrics.reset_index(drop = True).sort_values(["rmse"])


print(f'Melhor modelo: GARCH({best_params["p"]},{best_params["q"]}); AR({best_params["lags"]}) - RMSE: {best_rmse}')
print(metrics.head(10))

In [ ]:
gm_opt = arch_model(log_retornos["AGRO3.SA"].dropna(), 
                      mean = "AR", vol = "GARCH", 
                      p = best_params["p"], 
                      q = best_params["q"], 
                      lags = best_params["lags"], 
                      dist = "normal").fit(disp = 'off')

opt_forecasts = gm_opt.forecast(start=date.today(), method="simulation",
                                horizon=5,simulations = 100)
forecast_retornos = opt_forecasts.mean
forecast_variancia = opt_forecasts.variance
forecast_retornos.assign(acao = "AGRO3.SA").reset_index(drop = True)

## 6.3 Validação sequencial para todos os ativos

Estende-se a validação cruzada para todos os ativos, executando o grid search de
forma sequencial (loop). Para viabilizar a execução em tempo razoável, utiliza-se
uma busca aleatória (*randomized search*) com 10 combinações sorteadas da grade
completa, em vez de testar exaustivamente todas as combinações.

In [ ]:
# Inicialização
resultados_opt = pd.DataFrame()
forecasts_opt  = pd.DataFrame()
metrics        = pd.DataFrame()

# Rndomized Grid Search
random.seed(42) 
rand_hyper_grid = random.sample(hyper_grid, 10)
for col in tqdm(log_retornos.columns):
    
    best_rmse = float('inf')
    best_params = {}
    
    # Treino e teste
    treino = log_retornos[col][:-n_teste]
    teste = log_retornos[col][-n_teste:]

    for p, q, o, lags in rand_hyper_grid :               

        # Ajustar o modelo ARCH
        am = arch_model(treino, vol='GARCH', p=p, o = o, q=q, 
                        mean = "AR", lags = lags)
        res = am.fit(update_freq=0, disp='off', options={'maxiter': 500})

        # Gerar previsões para as 10 últimas observações
        previsoes = res.forecast(horizon=n_teste)

        # Calcular o RMSE nas 10 últimas observações
        rmse = round(np.sqrt(mean_squared_error(teste, previsoes.mean.values[-1])), 6)
        
        # Armazenar métricas
        metrics_temp = pd.DataFrame({"acao": [col], "p": [p], "o": [o], "q": [q], 
                                     "lags": [lags], "rmse": [rmse]})
        metrics      = pd.concat([metrics, metrics_temp])

        if rmse < best_rmse:
            best_rmse = rmse
            best_params = {'p': p, 'o': o, 'q': q, 'lags': lags}
    
    # Inicialização do modelo
    model = arch_model(log_retornos[col].dropna(), 
                        vol='GARCH', 
                        p=best_params["p"], 
                        o = best_params["o"], 
                        q=best_params["q"],
                        mean = "AR", 
                        lags = best_params["lags"])

    # Estimação
    res_opt = model.fit(update_freq = 0, disp = 'off')   
    
    # Previsões
    sim_forecasts = res_opt.forecast(start=date.today(), method="simulation", 
                                        simulations = 100,
                                       horizon=5)    
    
    # Armazennamento
    ## volatilidade prevista
    variance_temp = (sim_forecasts.variance
                     .assign(acao = col)
                     .reset_index(drop = True)
                     .melt(id_vars = "acao", value_name="v_prevista")
                     .drop(["variable"], axis = 1)
                     .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                     )
    ## retornos previstos
    returns_temp = (sim_forecasts.mean
                    .assign(acao = col)
                    .reset_index(drop = True)
                    .melt(id_vars = "acao", value_name="r_previstos")
                    .drop(["variable"], axis = 1)
                    .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                    )
    ## Merge
    forecasts_temp = variance_temp.merge(returns_temp, how = "left")
    ## Concat
    forecasts_opt  = pd.concat([forecasts_temp, forecasts_opt ])


metrics = metrics.reset_index(drop = True).sort_values(["acao", "rmse"])

print("Previsões do Modelo GARCH(1)\n",
      forecasts_opt.sort_values(["acao", "datas"], ascending=False).head(10)
      )

print(metrics.head())

## 6.4 Validação paralelizada para todos os ativos

Versão otimizada da seção anterior: utiliza-se `joblib.Parallel` para distribuir
o ajuste das combinações de hiperparâmetros entre múltiplos núcleos da CPU. A
função auxiliar `ajustar_modelo()` (definida em `utils.py`) encapsula o ajuste
individual e o cálculo do RMSE, retornando as métricas para consolidação.

In [ ]:
# Inicialização
resultados_opt = pd.DataFrame()
forecasts_opt  = pd.DataFrame()
metrics        = pd.DataFrame()

# Rndomized Grid Search
random.seed(42) 
rand_hyper_grid = random.sample(hyper_grid, 10)
for col in tqdm(log_retornos.columns):
    
    best_rmse = float('inf')
    best_params = {}
    
    # Treino e teste
    treino = log_retornos[col][:-n_teste]
    teste = log_retornos[col][-n_teste:]

    # Paralelização do ajuste
    res = Parallel(n_jobs=jobs, backend="loky")(
                             delayed(ajustar_modelo)(treino, teste, col, p, o, q, lags, 
                                                     n_teste=n_teste)
                             for p, o, q, lags in rand_hyper_grid
                            )
                            
    for metrics_temp, rmse, p, o, q, lags in res: # Desempacota os valores retornados
        if metrics_temp is not None:  
            metrics  = pd.concat([metrics, metrics_temp])
        if rmse < best_rmse:
            best_rmse = rmse
            best_params= {'p': p, 'o': o, 'q': q, 'lags': lags}  # Mantém a lógica dentro do loop
    
    # Inicialização do modelo
    model = arch_model(log_retornos[col].dropna(), 
                       vol='GARCH', 
                       p=best_params["p"], 
                       o = best_params["o"], 
                       q=best_params["q"],
                       mean = "AR", 
                       lags = best_params["lags"])

    # Estimação
    res_opt = model.fit(update_freq = 0, disp = 'off')   
    
    # Previsões
    sim_forecasts = res_opt.forecast(start=date.today(), method="simulation", 
                                     simulations = 100,
                                     horizon=5)    
    
    # Armazennamento
    ## volatilidade prevista
    variance_temp = (sim_forecasts.variance
                     .assign(acao = col)
                     .reset_index(drop = True)
                     .melt(id_vars = "acao", value_name="v_prevista")
                     .drop(["variable"], axis = 1)
                     .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                     )
    ## retornos previstos
    returns_temp = (sim_forecasts.mean
                    .assign(acao = col)
                    .reset_index(drop = True)
                    .melt(id_vars = "acao", value_name="r_previstos")
                    .drop(["variable"], axis = 1)
                    .assign(datas=pd.date_range(start=date.today() + timedelta(weeks=1), periods=5, freq='W'))
                    )
    ## Merge
    forecasts_temp = variance_temp.merge(returns_temp, how = "left")
    ## Concat
    forecasts_opt  = pd.concat([forecasts_temp, forecasts_opt ])


metrics = metrics.reset_index(drop = True).sort_values(["acao", "rmse"])
           

print("Previsões do Modelo GARCH(1)\n",
      forecasts_opt.sort_values(["acao", "datas"], ascending=False).head(10)
      )

print(metrics.head())

## Limitações

- A distribuição Normal pode não capturar adequadamente as caudas pesadas
  dos retornos financeiros (distribuições t-Student ou t-Student assimétrica seriam alternativas);
- O horizonte de previsão limitado a 5 semanas e o uso de 100 simulações
  podem ser insuficientes para capturar eventos extremos;
- A busca aleatória com apenas 10 combinações pode não encontrar o ótimo global.

## Extensões sugeridas

1. Testar distribuições alternativas (t-Student, t-Student assimétrica);
2. Incluir variáveis exógenas na equação da média (ex.: retorno do Ibovespa);
3. Ampliar o grid search com mais combinações e utilizar *time series split*
   em vez de uma única divisão treino-teste;
4. Comparar as previsões com medidas realizadas de volatilidade (ex.: *realized
   volatility* intraday) para avaliação mais precisa.